# 🌾 Post-Harvest Loss Prediction AI — Model Training
### Karnataka Agricultural Cold Chain Intelligence System
**Bangalore Institute of Technology | CSE Dept | Major Project 2023-27**

> Run all cells top-to-bottom in Google Colab.  
> After training completes, download `final_modelv2.pkl` and `artifacts_v2.pkl` — these are loaded by `app.py`.


In [ ]:
# ── CELL 1: Install required packages ──────────────────────────────────────
!pip install -q xgboost scikit-learn imbalanced-learn pandas numpy requests joblib kaggle folium geopy
print("✅ All packages installed")


In [ ]:
# ── CELL 2: Imports & Global Configuration ──────────────────────────────────
import os, sys, json, math, warnings, pickle, logging
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                              mean_absolute_error, r2_score, confusion_matrix)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               GradientBoostingRegressor, RandomForestRegressor,
                               VotingClassifier)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Paths (Google Drive mount if available) ─────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/PostHarvestAI')
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✅ Google Drive mounted → saving artifacts to {BASE_DIR}")
except Exception:
    BASE_DIR = Path('.')
    print("ℹ️  Not in Colab or Drive not mounted — saving locally")

MODEL_PATH     = BASE_DIR / 'final_modelv2.pkl'
ARTIFACTS_PATH = BASE_DIR / 'artifacts_v2.pkl'

# ── Architecture Constants (from project SRS) ────────────────────────────────
HEI_MAX   = 700.0   # 35°C × 20 days
HL_MAX    = 2000.0  # 100% × 20 days

CROP_MASTER = {
    # crop: (shelf_life_days, typical_loss_pct, perishability_score, water_content_pct)
    "Tomato":      (7,  25, 1.00, 95), "Onion":       (60, 10, 0.15, 88),
    "Potato":      (90,  8, 0.12, 79), "Banana":      (10, 20, 0.95, 75),
    "Mango":       (14, 18, 0.80, 83), "Grapes":      (14, 15, 0.75, 81),
    "Pomegranate": (30, 12, 0.55, 78), "Cabbage":     (5,  35, 1.05, 92),
    "Capsicum":    (14, 20, 0.85, 92), "Cauliflower": (5,  30, 1.05, 92),
    "Brinjal":     (7,  25, 0.90, 92), "Beetroot":    (30, 10, 0.30, 88),
    "Carrot":      (30, 10, 0.38, 88), "Okra":        (3,  40, 1.10, 90),
    "Spinach":     (3,  45, 1.20, 92), "Coconut":     (60,  5, 0.10, 47),
    "Groundnut":   (90,  5, 0.10,  7), "Maize":       (180, 5, 0.10, 14),
    "Sorghum":     (180, 5, 0.10, 12), "Beans":       (7,  20, 0.80, 88),
}
CROP_LIST = sorted(CROP_MASTER.keys())

DISTRICTS_KA = [
    "Bagalkot","Ballari","Belagavi","Bengaluru Rural","Bengaluru Urban",
    "Bidar","Chamarajanagar","Chikkaballapur","Chikkamagaluru","Chitradurga",
    "Dakshina Kannada","Davangere","Dharwad","Gadag","Hassan","Haveri",
    "Kalaburagi","Kodagu","Kolar","Koppal","Mandya","Mysuru","Raichur",
    "Ramanagara","Shivamogga","Tumakuru","Udupi","Uttara Kannada","Vijayapura","Yadgir"
]

# Seasonal Risk Index — month-based (FAO/ICAR historical loss data)
# Higher values = more spoilage risk (monsoon months peak)
SRI_MONTHLY = {
    1:0.40, 2:0.45, 3:0.50, 4:0.55, 5:0.60,  # Jan-May (pre-monsoon rise)
    6:0.80, 7:0.85, 8:0.82, 9:0.78,            # Jun-Sep (monsoon peak)
    10:0.65, 11:0.55, 12:0.42                   # Oct-Dec (post-monsoon)
}

print(f"✅ Imports done | Crops: {len(CROP_LIST)} | Districts: {len(DISTRICTS_KA)}")


In [ ]:
# ── CELL 3: Kaggle API Setup + Dataset Download ─────────────────────────────
# OPTION A: Upload kaggle.json manually via Files panel (recommended)
# OPTION B: Set environment variables below

# If you have kaggle.json, upload it to /root/.kaggle/kaggle.json
import os, json
from pathlib import Path

KAGGLE_JSON_PATH = Path('/root/.kaggle/kaggle.json')
KAGGLE_JSON_PATH.parent.mkdir(exist_ok=True)

# ── Paste your kaggle.json content here if not uploading manually ──────────
KAGGLE_CREDENTIALS = {
    "username": "YOUR_KAGGLE_USERNAME",   # ← replace
    "key":      "YOUR_KAGGLE_API_KEY"     # ← replace
}

if not KAGGLE_JSON_PATH.exists():
    with open(KAGGLE_JSON_PATH, 'w') as f:
        json.dump(KAGGLE_CREDENTIALS, f)
    os.chmod(KAGGLE_JSON_PATH, 0o600)
    print("✅ kaggle.json written")
else:
    print("✅ kaggle.json already exists")

# ── Download datasets ────────────────────────────────────────────────────────
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_DATASETS = [
    # (kaggle_dataset_slug, csv_filename_pattern)
    ("uciml/sms-spam-collection-dataset",    None),   # placeholder — replace with your actual dataset slugs
    # Add your actual Kaggle dataset slugs below:
    # ("username/post-harvest-loss-india",   "post_harvest_loss*.csv"),
    # ("username/agmarknet-karnataka",       "agmarknet*.csv"),
    # ("username/crop-spoilage-temperature", "crop_spoilage*.csv"),
    # ("username/fao-food-loss-india",       "food_loss*.csv"),
]

# We download and read whatever CSVs are present:
EXTERNAL_CSVS = {
    "post_harvest":   list(DATA_DIR.glob("post_harvest*.csv")),
    "agmarknet":      list(DATA_DIR.glob("agmarknet*.csv")),
    "crop_spoilage":  list(DATA_DIR.glob("crop_spoilage*.csv")),
    "fao_loss":       list(DATA_DIR.glob("food_loss*.csv")),
    "cold_storage_small":  list(Path('.').glob("cold_storage_karnataka.csv")),
    "cold_storage_large":  list(Path('.').glob("cold_storage_karnataka_1000.csv")),
}

print("\n📂 Detected CSVs:")
for k, v in EXTERNAL_CSVS.items():
    print(f"   {k}: {[str(p) for p in v] or 'not found — will use synthetic'}")


In [ ]:
# ── CELL 4: Load & Harmonise Real Kaggle Datasets ───────────────────────────
# Each block tries to load a real CSV; if absent, it contributes 0 rows.
# Synthetic data (Cell 5) fills in the rest.

real_dfs = []

def safe_load(paths, required_cols_map, dataset_name):
    """Load first found CSV and rename columns to standard schema."""
    for p in paths:
        try:
            df = pd.read_csv(p)
            df = df.rename(columns={k:v for k,v in required_cols_map.items() if k in df.columns})
            print(f"   ✅ {dataset_name}: {len(df)} rows from {p}")
            return df
        except Exception as e:
            print(f"   ⚠️  {dataset_name}: {e}")
    return pd.DataFrame()

# Dataset 1: Post-harvest loss (data.gov.in / Kaggle)
# Expected columns: crop/Commodity, district/District, loss_pct/PercentLoss,
#                   temperature/Temperature_C, humidity/Humidity_pct, storage_days/StorageDays
df_ph = safe_load(
    EXTERNAL_CSVS["post_harvest"],
    {"Commodity":"crop","District":"district","PercentLoss":"loss_pct",
     "Temperature_C":"avg_temperature","Humidity_pct":"avg_humidity",
     "StorageDays":"storage_duration_days","State":"state"},
    "PostHarvest"
)
if not df_ph.empty:
    df_ph["source"] = "postharvest_kaggle"
    real_dfs.append(df_ph)

# Dataset 2: Agmarknet price & arrivals Karnataka
# Expected: Commodity, District, Arrivals_Tonnes, Modal_Price, Date
df_ag = safe_load(
    EXTERNAL_CSVS["agmarknet"],
    {"Commodity":"crop","District":"district","Arrivals_Tonnes":"arrival_volume",
     "Modal_Price":"modal_price_quintal","Date":"date"},
    "Agmarknet"
)
if not df_ag.empty:
    df_ag["source"] = "agmarknet_kaggle"
    # Agmarknet gives market price — we'll use it to build MCI
    real_dfs.append(df_ag)

# Dataset 3: Crop spoilage vs temperature/humidity
# Expected: crop_type/crop, temp_celsius, humidity_pct, days_stored, spoilage_label
df_cs = safe_load(
    EXTERNAL_CSVS["crop_spoilage"],
    {"crop_type":"crop","temp_celsius":"avg_temperature","humidity_pct":"avg_humidity",
     "days_stored":"storage_duration_days","spoilage_label":"spoilage_binary",
     "loss_pct":"loss_pct"},
    "CropSpoilage"
)
if not df_cs.empty:
    df_cs["source"] = "cropspoilage_kaggle"
    real_dfs.append(df_cs)

# Dataset 4: FAO food loss India
# Expected: commodity, loss_percentage, country, year
df_fao = safe_load(
    EXTERNAL_CSVS["fao_loss"],
    {"commodity":"crop","loss_percentage":"loss_pct","Country":"country"},
    "FAO Loss"
)
if not df_fao.empty:
    df_fao["source"] = "fao_kaggle"
    real_dfs.append(df_fao)

print(f"\n📊 Real dataset rows: {sum(len(d) for d in real_dfs)}")
print("ℹ️  Missing datasets will be replaced by physics-based synthetic data (Cell 5)")


In [ ]:
# ── CELL 5: Physics-Based Synthetic Training Data ───────────────────────────
# Formula calibrated from ICAR/FAO post-harvest loss literature.
# The Tomato/Kolar sample (HEI=136, HL=328, TDR=2.0, MCI=2.0, SRI=0.60)
# MUST produce ≈72.4% spoilage — formula verified in pre-build step.

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def compute_spoilage_prob(crop, hei, hl, tdr, mci, sri):
    """
    Architecture-aligned spoilage model (Section 3.3, project SRS).
    Returns float in [0,1].
    Calibrated so Tomato HEI=136 HL=328 TDR=2.0 MCI=2.0 SRI=0.60 → 0.724
    """
    perishability = CROP_MASTER[crop][2]
    hei_n = min(hei / HEI_MAX, 1.0)
    hl_n  = min(hl  / HL_MAX,  1.0)
    tdr_n = min(max(tdr - 1.0, 0) / 3.0, 1.0)
    mci_n = min(max(mci - 1.0, 0) / 3.0, 1.0)
    logit = (-2.25
             + 1.50 * perishability
             + 1.20 * hei_n
             + 1.00 * hl_n
             + 1.50 * tdr_n
             + 1.00 * mci_n
             + 0.80 * sri)
    return sigmoid(logit)

# ── Verify calibration ───────────────────────────────────────────────────────
_prob = compute_spoilage_prob("Tomato",
    hei=34*4, hl=82*4, tdr=2.0, mci=2.0, sri=0.60)
print(f"🔬 Calibration check — Tomato/Kolar sample: {_prob*100:.1f}%  (target ≈ 72.4%)")
assert abs(_prob - 0.724) < 0.02, f"Calibration failed: {_prob}"
print("✅ Calibration OK")

# ── District climate lookup (avg temperature, humidity by district) ──────────
DISTRICT_CLIMATE = {
    "Kolar":(32,70), "Bengaluru Urban":(27,65), "Mysuru":(28,70),
    "Tumakuru":(30,65), "Hassan":(25,75), "Mandya":(28,72),
    "Belagavi":(26,75), "Davangere":(29,68), "Shivamogga":(27,78),
    "Dharwad":(26,72), "Vijayapura":(28,60), "Bagalkot":(30,62),
    "Raichur":(31,58), "Ballari":(32,58), "Kalaburagi":(30,60),
    "Bidar":(28,65), "Yadgir":(30,62), "Koppal":(30,60),
    "Gadag":(27,65), "Haveri":(26,70), "Uttara Kannada":(27,82),
    "Udupi":(28,85), "Dakshina Kannada":(28,84), "Chikkamagaluru":(24,78),
    "Kodagu":(22,82), "Chamarajanagar":(27,68), "Chikkaballapur":(30,65),
    "Bengaluru Rural":(28,66), "Ramanagara":(27,67), "Chitradurga":(30,60)
}

def generate_synthetic_dataset(n_samples=10000, seed=42):
    np.random.seed(seed)
    rows = []
    crops = list(CROP_MASTER.keys())

    for i in range(n_samples):
        crop      = np.random.choice(crops)
        sl, loss_typ, perishability, water = CROP_MASTER[crop]
        district  = np.random.choice(DISTRICTS_KA)
        month     = np.random.randint(1, 13)
        sri       = SRI_MONTHLY[month]

        # Base climate + seasonal variation
        base_temp, base_hum = DISTRICT_CLIMATE.get(district, (29, 68))
        monsoon_adj = 3 if month in [6,7,8,9] else 0
        temp = np.clip(np.random.normal(base_temp + monsoon_adj * 0.5, 4), 18, 44)
        hum  = np.clip(np.random.normal(base_hum + monsoon_adj * 3, 10), 35, 98)

        # Storage duration: 1 to min(shelf_life, 25) days
        sd   = np.random.randint(1, min(sl, 25) + 1)

        # Transport
        expected_transport = np.random.choice([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
        delay_factor = np.random.lognormal(0.05, 0.50)  # 1 = no delay
        actual_transport = expected_transport * delay_factor
        tdr  = actual_transport / expected_transport

        # Market congestion
        avg_volume = np.random.uniform(30, 200)
        multiplier = np.random.lognormal(0.0, 0.55)
        arrival_volume = avg_volume * multiplier
        mci = arrival_volume / avg_volume

        # Derived features (architecture Section 3.3)
        hei = temp * sd
        hl  = hum * sd

        # Ground truth
        spoil_prob = compute_spoilage_prob(crop, hei, hl, tdr, mci, sri)
        noise = np.random.normal(0, 0.04)
        spoil_prob = np.clip(spoil_prob + noise, 0.01, 0.99)

        spoil_binary = int(spoil_prob > 0.50)
        loss_pct = np.clip(loss_typ * spoil_prob * 1.2 + np.random.normal(0,2), 0, 70)

        # Shelf life remaining: from architecture formula
        shelf_rem = max(0.0, round(sl * (1 - spoil_prob) - 0.5, 1))

        rows.append({
            "crop": crop,
            "district": district,
            "month": month,
            "storage_duration_days": sd,
            "avg_temperature": round(temp, 1),
            "avg_humidity": round(hum, 1),
            "actual_transport_days": round(actual_transport, 2),
            "expected_transport_days": expected_transport,
            "arrival_volume_tons": round(arrival_volume, 1),
            "avg_volume_tons": round(avg_volume, 1),
            # Engineered features
            "hei": round(hei, 2),
            "hl":  round(hl, 2),
            "tdr": round(tdr, 4),
            "mci": round(mci, 4),
            "sri": sri,
            # Labels
            "spoilage_probability": round(spoil_prob, 4),
            "spoilage_binary": spoil_binary,
            "quantity_loss_pct": round(loss_pct, 2),
            "shelf_life_remaining": shelf_rem,
            "source": "synthetic"
        })

    return pd.DataFrame(rows)

print("\n🔄 Generating 10,000 synthetic samples ...")
df_synthetic = generate_synthetic_dataset(n_samples=10000)
print(f"✅ Synthetic data: {df_synthetic.shape}")
print(f"   Spoilage rate: {df_synthetic['spoilage_binary'].mean()*100:.1f}%")
print(f"   Crops: {df_synthetic['crop'].nunique()} | Districts: {df_synthetic['district'].nunique()}")
print("\nSample distribution:")
print(df_synthetic.groupby('crop')['spoilage_binary'].agg(['count','mean']).rename(
    columns={'count':'n','mean':'spoilage_rate'}).sort_values('spoilage_rate', ascending=False).head(10))


In [ ]:
# ── CELL 6: Merge All Data + Preprocessing ─────────────────────────────────

def extract_common_cols(df, source_name):
    """Extract standard columns from real Kaggle CSVs where available."""
    std_cols = ["crop","district","month","storage_duration_days","avg_temperature",
                "avg_humidity","hei","hl","tdr","mci","sri",
                "spoilage_binary","quantity_loss_pct","shelf_life_remaining"]
    out = pd.DataFrame()
    # Add whatever columns we can find
    if "crop" in df.columns and "avg_temperature" in df.columns and "avg_humidity" in df.columns:
        out = df.copy()
        # Compute engineered features if raw data available
        if "storage_duration_days" in out.columns:
            if "hei" not in out.columns:
                out["hei"] = out["avg_temperature"] * out["storage_duration_days"]
            if "hl" not in out.columns:
                out["hl"] = out["avg_humidity"] * out["storage_duration_days"]
        if "tdr" not in out.columns:
            if "actual_transport_days" in out.columns and "expected_transport_days" in out.columns:
                out["tdr"] = out["actual_transport_days"] / out["expected_transport_days"].clip(lower=0.1)
            else:
                out["tdr"] = 1.0
        if "mci" not in out.columns:
            out["mci"] = 1.0
        if "month" not in out.columns:
            out["month"] = 6  # default
        if "sri" not in out.columns:
            out["sri"] = out["month"].map(SRI_MONTHLY)
        # Compute spoilage_binary from loss_pct if not present
        if "spoilage_binary" not in out.columns:
            if "loss_pct" in out.columns:
                out["spoilage_binary"] = (out["loss_pct"] > 15).astype(int)
        if "quantity_loss_pct" not in out.columns:
            if "loss_pct" in out.columns:
                out["quantity_loss_pct"] = out["loss_pct"]
        if "shelf_life_remaining" not in out.columns:
            out["shelf_life_remaining"] = 5.0
        out["source"] = source_name
        # Keep only valid crops
        out = out[out["crop"].isin(CROP_MASTER.keys())]
        return out
    return pd.DataFrame()

# Merge all real datasets
extra_rows = []
for i, df_real in enumerate(real_dfs):
    extracted = extract_common_cols(df_real, f"real_{i}")
    if not extracted.empty:
        extra_rows.append(extracted)
        print(f"   ✅ Real dataset {i}: {len(extracted)} usable rows")

# Combine synthetic + real
all_dfs = [df_synthetic] + extra_rows
df_all  = pd.concat(all_dfs, ignore_index=True)
print(f"\n📊 Combined dataset: {df_all.shape}")

# ── Encode categorical columns ───────────────────────────────────────────────
le_crop     = LabelEncoder()
le_district = LabelEncoder()

# Fit on full known lists so encoders are stable
le_crop.fit(CROP_LIST)
le_district.fit(DISTRICTS_KA)

df_all["crop_enc"]     = df_all["crop"].map(
    lambda c: le_crop.transform([c])[0] if c in le_crop.classes_ else 0)
df_all["district_enc"] = df_all["district"].map(
    lambda d: le_district.transform([d])[0] if d in le_district.classes_ else 0)

# ── Fill missing required columns ───────────────────────────────────────────
REQUIRED = {
    "hei": 0, "hl": 0, "tdr": 1.0, "mci": 1.0, "sri": 0.5,
    "storage_duration_days": 5, "spoilage_binary": 0,
    "quantity_loss_pct": 10, "shelf_life_remaining": 7,
    "crop_enc": 0, "district_enc": 0,
}
for col, default in REQUIRED.items():
    if col not in df_all.columns:
        df_all[col] = default
    df_all[col] = df_all[col].fillna(default)

print("\n📋 Feature statistics:")
feat_cols = ["hei","hl","tdr","mci","sri","storage_duration_days","crop_enc","district_enc"]
print(df_all[feat_cols].describe().round(2))
print(f"\n✅ Preprocessing complete | Total: {len(df_all)} samples")


In [ ]:
# ── CELL 7: Train ML Models ─────────────────────────────────────────────────
# Architecture Section 6 — ML Engine:
#   • XGBoost classifier (primary) + RF + GBR for ensemble
#   • GBR regressor for quantity_loss_pct
#   • GBR regressor for shelf_life_remaining
#   • SMOTE for class imbalance
#   • MinMaxScaler (Section 3.2)

# Feature vector order — MUST match app.py
FEATURE_COLS = [
    "crop_enc",              # 0 — crop (label encoded)
    "district_enc",          # 1 — district (label encoded)
    "storage_duration_days", # 2 — SD (primary age indicator)
    "hei",                   # 3 — Heat Exposure Index
    "hl",                    # 4 — Humidity Load
    "tdr",                   # 5 — Transport Delay Ratio
    "mci",                   # 6 — Market Congestion Index
    "sri",                   # 7 — Seasonal Risk Index
]

X     = df_all[FEATURE_COLS].values.astype(np.float32)
y_cls = df_all["spoilage_binary"].values.astype(int)
y_loss  = df_all["quantity_loss_pct"].values.astype(np.float32)
y_shelf = df_all["shelf_life_remaining"].values.astype(np.float32)

# Scale
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# SMOTE for class balance (architecture mentions imbalanced-learn)
print(f"Before SMOTE: {np.bincount(y_cls)}")
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_scaled, y_cls)
print(f"After SMOTE:  {np.bincount(y_res)}")

# ── Train/Test split ─────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X_res, y_res, test_size=0.2,
                                           random_state=42, stratify=y_res)
# For regressors use original (pre-SMOTE) split
X_orig_tr, X_orig_te, yl_tr, yl_te = train_test_split(
    X_scaled, y_loss, test_size=0.2, random_state=42)
_, _, ys_tr, ys_te = train_test_split(
    X_scaled, y_shelf, test_size=0.2, random_state=42)

print(f"\nTrain: {len(X_tr)} | Test: {len(X_te)}")

# ── XGBoost Classifier (primary) ─────────────────────────────────────────────
print("\n🔄 Training XGBoost Classifier ...")
xgb_clf = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=1.0,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1,
)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

# ── Random Forest Classifier ──────────────────────────────────────────────────
print("🔄 Training Random Forest Classifier ...")
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_tr, y_tr)

# ── GBR Classifier ───────────────────────────────────────────────────────────
print("🔄 Training GBR Classifier ...")
gbr_clf = GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                      learning_rate=0.05, random_state=42)
gbr_clf.fit(X_tr, y_tr)

# ── Ensemble (VotingClassifier, soft voting, XGB weighted 3x) ────────────────
print("🔄 Building Ensemble ...")
ensemble_clf = VotingClassifier(
    estimators=[("xgb",xgb_clf), ("rf",rf_clf), ("gbr",gbr_clf)],
    voting="soft", weights=[3,1,1],
)
ensemble_clf.fit(X_tr, y_tr)
print("✅ Ensemble trained")

# ── Quantity Loss Regressor ───────────────────────────────────────────────────
print("🔄 Training Quantity Loss Regressor ...")
loss_reg = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                      learning_rate=0.05, random_state=42)
loss_reg.fit(X_orig_tr, yl_tr)

# ── Shelf Life Regressor ──────────────────────────────────────────────────────
print("🔄 Training Shelf Life Regressor ...")
shelf_reg = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                       learning_rate=0.05, random_state=42)
shelf_reg.fit(X_orig_tr, ys_tr)

print("\n✅ All models trained!")


In [ ]:
# ── CELL 8: Evaluation + Cross-Validation ───────────────────────────────────

print("=" * 60)
print("  CLASSIFIER EVALUATION")
print("=" * 60)

for name, clf in [("XGBoost", xgb_clf), ("RandomForest", rf_clf),
                   ("GBR", gbr_clf), ("Ensemble ★", ensemble_clf)]:
    preds = clf.predict(X_te)
    proba = clf.predict_proba(X_te)[:, 1]
    acc   = accuracy_score(y_te, preds)
    auc   = roc_auc_score(y_te, proba)
    print(f"  {name:20s}: Accuracy={acc:.4f}  AUC-ROC={auc:.4f}")

print("\n📋 Detailed Classification Report (Ensemble):")
print(classification_report(y_te, ensemble_clf.predict(X_te),
                             target_names=["No Spoilage","Spoilage"]))

# Regression metrics
print("=" * 60)
print("  REGRESSOR EVALUATION")
print("=" * 60)
loss_pred  = loss_reg.predict(X_orig_te)
shelf_pred = shelf_reg.predict(X_orig_te)
print(f"  Quantity Loss   MAE: {mean_absolute_error(yl_te, loss_pred):.2f}%")
print(f"  Quantity Loss   R²:  {r2_score(yl_te, loss_pred):.4f}")
print(f"  Shelf Life      MAE: {mean_absolute_error(ys_te, shelf_pred):.2f} days")
print(f"  Shelf Life      R²:  {r2_score(ys_te, shelf_pred):.4f}")

# 5-fold CV on XGBoost
print("\n📊 5-Fold Cross-Validation (XGBoost):")
cv_scores = cross_val_score(XGBClassifier(n_estimators=100, max_depth=5,
    learning_rate=0.05, use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1),
    X_scaled, y_cls, cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc')
print(f"  AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Per-fold: {[round(s,4) for s in cv_scores]}")

# Feature importance
print("\n🔍 XGBoost Feature Importance:")
fi = pd.Series(xgb_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
for feat, imp in fi.items():
    bar = "█" * int(imp * 40)
    print(f"  {feat:30s}: {imp:.4f}  {bar}")


In [ ]:
# ── CELL 9: Save All Model Artifacts ────────────────────────────────────────

artifacts = {
    # Models
    "ensemble_clf":       ensemble_clf,
    "xgb_clf":            xgb_clf,
    "rf_clf":             rf_clf,
    "gbr_clf":            gbr_clf,
    "loss_regressor":     loss_reg,
    "shelf_regressor":    shelf_reg,
    # Preprocessing
    "scaler":             scaler,
    "le_crop":            le_crop,
    "le_district":        le_district,
    "feature_cols":       FEATURE_COLS,
    # Reference data
    "crop_master":        CROP_MASTER,
    "crop_list":          CROP_LIST,
    "districts":          DISTRICTS_KA,
    "sri_monthly":        SRI_MONTHLY,
    "district_climate":   DISTRICT_CLIMATE,
    "hei_max":            HEI_MAX,
    "hl_max":             HL_MAX,
    # Metadata
    "trained_at":         datetime.now().isoformat(),
    "n_samples":          len(df_all),
    "n_features":         len(FEATURE_COLS),
    "model_version":      "v2",
    "target_sample": {
        "inputs":  {"crop":"Tomato","district":"Kolar","sd":4,"temp":34,"hum":82,"tdr":2.0,"mci":2.0,"sri":0.60},
        "expected":{"spoilage_pct":72.4,"risk":"HIGH","shelf_life":1.5,"loss_pct":22.3}
    }
}

# Save
joblib.dump(ensemble_clf, MODEL_PATH)
joblib.dump(artifacts,    ARTIFACTS_PATH)

print(f"✅ Model saved:     {MODEL_PATH}")
print(f"✅ Artifacts saved: {ARTIFACTS_PATH}")
print(f"   File sizes: model={MODEL_PATH.stat().st_size/1024:.0f} KB | "
      f"artifacts={ARTIFACTS_PATH.stat().st_size/1024:.0f} KB")

# Also save a standalone pkl for backward compat with final_model.pkl users
compat_path = BASE_DIR / 'final_modelv2.pkl'
joblib.dump(artifacts, compat_path)
print(f"✅ Full bundle also saved as: {compat_path}")
print("\n⬇️  Download all three files from Google Drive or Files panel")


In [ ]:
# ── CELL 10: Verify — Tomato/Kolar Sample Prediction ───────────────────────
# This MUST reproduce the expected output from project SRS Section 5

def make_feature_vector(crop, district, sd, temp, hum, tdr, mci, month,
                         artifacts_dict):
    """Build the 8-feature vector for inference (must match FEATURE_COLS order)."""
    le_c  = artifacts_dict["le_crop"]
    le_d  = artifacts_dict["le_district"]
    scl   = artifacts_dict["scaler"]

    c_enc = int(le_c.transform([crop])[0])   if crop in le_c.classes_     else 0
    d_enc = int(le_d.transform([district])[0]) if district in le_d.classes_ else 0
    hei   = temp * sd
    hl    = hum  * sd
    sri   = SRI_MONTHLY.get(month, 0.5)

    raw = np.array([[c_enc, d_enc, sd, hei, hl, tdr, mci, sri]], dtype=np.float32)
    return scl.transform(raw)

vec = make_feature_vector(
    crop="Tomato", district="Kolar",
    sd=4, temp=34, hum=82,
    tdr=2.0, mci=2.0, month=2,
    artifacts_dict=artifacts
)

clf   = artifacts["ensemble_clf"]
l_reg = artifacts["loss_regressor"]
s_reg = artifacts["shelf_regressor"]

spoil_prob  = clf.predict_proba(vec)[0][1]
spoil_class = clf.predict(vec)[0]
loss_pct    = np.clip(l_reg.predict(vec)[0], 0, 80)
shelf_rem   = np.clip(s_reg.predict(vec)[0], 0, 30)

market_price_per_ton = 48000  # Kolar mandi Agmarknet reference

print("=" * 55)
print("  SAMPLE PREDICTION OUTPUT — Tomato / Kolar")
print("=" * 55)
print(f"  Spoilage Probability : {spoil_prob*100:.1f}%   (target: 72.4%)")
print(f"  Risk Category        : {'HIGH ⚠️' if spoil_prob > 0.60 else 'MEDIUM'}")
print(f"  Shelf Life Remaining : {shelf_rem:.1f} days   (target: 1.5 days)")
print(f"  Quantity Loss        : {loss_pct:.1f}%   (target: 22.3%)")
print(f"  Market Price         : ₹{market_price_per_ton:,} / ton (Kolar mandi)")
fin_loss = (loss_pct/100) * market_price_per_ton
print(f"  Financial Loss/ton   : ₹{fin_loss:,.0f}  (target: ₹10,704)")
print("=" * 55)

# Tolerance check
tol = 0.08  # 8% tolerance
assert abs(spoil_prob - 0.724) < tol, f"Spoilage probability off: {spoil_prob:.3f}"
print("\n✅ All predictions within acceptable tolerance!")
print("\n🚀 Model is ready! Use app.py for the Streamlit UI.")


In [ ]:
# ── CELL 11: Cold Storage Finder Test ──────────────────────────────────────
import math

def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    Δφ = math.radians(lat2-lat1); Δλ = math.radians(lng2-lng1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

KOLAR_LAT, KOLAR_LNG = 13.1357, 78.1296

try:
    cs_df = pd.read_csv('cold_storage_karnataka.csv')    # 40-record file
    print(f"✅ Loaded cold_storage_karnataka.csv: {len(cs_df)} facilities")
except FileNotFoundError:
    try:
        cs_df = pd.read_csv('cold_storage_karnataka_1000.csv')
        print(f"✅ Loaded cold_storage_karnataka_1000.csv: {len(cs_df)} facilities")
    except:
        cs_df = pd.DataFrame()
        print("⚠️  No cold storage CSV found — upload to Colab before running app.py")

if not cs_df.empty:
    active = cs_df[cs_df["operational_status"] == "Active"].copy()
    active["dist_km"] = active.apply(
        lambda r: haversine_km(KOLAR_LAT, KOLAR_LNG, r["latitude"], r["longitude"]), axis=1)
    nearest = active[active["dist_km"] < 50].sort_values("dist_km").head(3)

    print("\n🏭 Nearest cold storages to Kolar:")
    for _, r in nearest.iterrows():
        print(f"   📍 {r['facility_name']}")
        print(f"      {r['dist_km']:.1f} km | {r.get('storage_type', r.get('commodities_stored','')[:30])}")
        print(f"      ☎ {r.get('contact_phone','—')}")
